# Rebuild `Wind_bow_shock_database.h5` from the crossing list

The HDF5 file is the crossing list `bs_crossings_V02.xlsx` (sheet `bs_crossings_V02`) stored as a pandas
table under the key `all`. Run this notebook whenever the list changes.

It reads the existing file first and writes the new one with the same key, column order, column types and
storage format, so code that reads the old file reads the new one unchanged. Each file it replaces is archived
in `bow_shock/00_old_versions/` (see its README.md), and the result is read back and compared with the list before the
notebook reports success.

- **Input:** the working copy of the list, `bow_shock/bs_crossings_V02.xlsx`, which every MagCarto notebook
  reads. `sync_release_to_repo.ipynb` copies it into the repository.
- **Outputs:** `database_files/Wind_bow_shock_database.h5` in the repository and
  `Figures/Wind_bow_shock_database.h5`. The first is written, the second is a byte copy of it.
- **Needs:** pandas, openpyxl and PyTables (`pip install tables`). No network access.

## 1. Configuration

In [ ]:
from pathlib import Path
import datetime as dt
import hashlib
import shutil

import pandas as pd

import json

# Machine-specific locations come from local_paths.json beside this notebook. That file is not part of the
# repository: copy local_paths.example.json and fill it in. Without it, BOW_SHOCK is the folder above this one.
_local_file = Path("local_paths.json")
LOCAL = json.loads(_local_file.read_text(encoding="utf-8")) if _local_file.exists() else {}
BOW_SHOCK = Path(LOCAL.get("bow_shock", Path.cwd().parent))
REPO      = BOW_SHOCK / "Wind_Bow_Shock_database"

CROSSINGS = BOW_SHOCK / "bs_crossings_V02.xlsx"      # the working copy; the repository gets it from the sync
SHEET     = "bs_crossings_V02"
TARGETS   = [REPO / "database_files" / "Wind_bow_shock_database.h5",   # written
             BOW_SHOCK / "Figures" / "Wind_bow_shock_database.h5"]       # copied from the first
KEY       = "all"
COLUMNS   = ["pass", "SC direction", "time", "direction",
             "start time", "end time", "start time zoom", "end time zoom"]

WRITE   = True     # False builds and compares only, and writes nothing
ARCHIVE = BOW_SHOCK / "00_old_versions"   # replaced files are archived here (see its README.md)


def md5(path):
    return hashlib.md5(Path(path).read_bytes()).hexdigest()


for p in [CROSSINGS, *TARGETS]:
    print(f"{'ok ' if p.exists() else 'MISSING':<8} {p}")

## 2. Read the crossing list

In [ ]:
x = pd.read_excel(CROSSINGS, sheet_name=SHEET)
assert list(x.columns[:len(COLUMNS)]) == COLUMNS, f"unexpected columns: {list(x.columns)}"
x = x.loc[x["pass"].notna(), COLUMNS].reset_index(drop=True)
x["pass"] = x["pass"].astype(int)

print(f"{len(x)} crossings, passes {x['pass'].min()}-{x['pass'].max()}, "
      f"{x.groupby(['pass', 'SC direction']).ngroups} pass legs")
print("blank cells per column:", {c: int(n) for c, n in x.isna().sum().items()})

## 3. Look at the existing file

Records each existing file's keys, storage format, compression and column types. Nothing is written here. The
notebook stops if a file holds any key other than `all`, because writing would delete it.

In [ ]:
def describe(path):
    """Keys, storage format, compression and the stored table of an existing HDF5 file."""
    with pd.HDFStore(path, mode="r") as store:
        keys = store.keys()
        storer = store.get_storer(KEY) if f"/{KEY}" in keys else None
        fmt = "table" if (storer is not None and storer.is_table) else "fixed"
        complevel, complib = 0, None
        try:
            node = store.get_node(f"/{KEY}")
            leaf = next(iter(node._f_walknodes("Leaf")))
            complevel, complib = leaf.filters.complevel, leaf.filters.complib
        except Exception:
            pass
    table = pd.read_hdf(path, key=KEY) if storer is not None else None
    return dict(keys=keys, fmt=fmt, complevel=complevel, complib=complib, table=table)


existing = {}
for t in TARGETS:
    if not t.exists():
        print(f"{t}: not there yet, will be created")
        continue
    d = existing[t] = describe(t)
    print(f"{t}\n   keys {d['keys']}, format '{d['fmt']}', complevel {d['complevel']} {d['complib'] or ''}, "
          f"md5 {md5(t)}")
    if d["table"] is not None:
        print(f"   {len(d['table'])} rows, columns {list(d['table'].columns)}")
        print("   types:", {c: str(v) for c, v in d["table"].dtypes.items()})
    other = [k for k in d["keys"] if k != f"/{KEY}"]
    assert not other, f"{t} also holds {other}; writing would delete them"

## 4. Build the new table with the old column types

Datetime columns stay datetime, integer columns stay integer, and text stays text. If no file exists yet, the
types are those of the crossing list as read.

In [ ]:
ref = next((d for d in existing.values() if d["table"] is not None), None)
new = x.copy()
if ref is not None:
    old = ref["table"]
    assert list(old.columns) == COLUMNS, f"existing columns differ: {list(old.columns)}"
    for c in COLUMNS:
        kind = old[c].dtype.kind
        if kind == "M":
            new[c] = pd.to_datetime(new[c])
        elif kind in "iu":
            new[c] = new[c].astype(old[c].dtype)
        elif kind == "f":
            new[c] = new[c].astype(float)
        else:
            new[c] = new[c].astype(object)
    if type(old.index) is not type(new.index):
        print(f"note: the existing index is {type(old.index).__name__}, the new one {type(new.index).__name__}")
print("new types:", {c: str(v) for c, v in new.dtypes.items()})

## 5. What changes

A cell-by-cell comparison of the new table with the existing file. Against a file built from the list as it
stood before 8 October 2026 this shows 1,993 changed cells. Five are the pass 8 inbound directions. Eight
are window values for passes 5, 6, 12 and 68 inbound, corrected or filled from the 2D renders. The other 1,980
are window values repeated down their legs. Each one is listed in
`bs_crossings_V02_change_log_20261008_103312.csv`.

In [ ]:
if ref is None:
    print("no existing file to compare with")
else:
    a, b = ref["table"].reset_index(drop=True), new.reset_index(drop=True)
    print("shape: existing", a.shape, "new", b.shape)
    if a.shape == b.shape:
        differs = ~((a == b) | (a.isna() & b.isna()))
        print(f"{int(differs.values.sum())} cells differ:", {c: int(n) for c, n in differs.sum().items() if n})
        rows = differs.any(axis=1)
        print(f"{int(rows.sum())} rows affected; the first 15:")
        print(pd.concat({"existing": a[rows], "new": b[rows]}, axis=1).head(15).to_string())

## 6. Write and check

Each existing target is archived first, in `00_old_versions/<date>_<time>_build_hdf5_from_crossings/` under its
original path, with a row in `00_old_versions/INDEX.csv`. The first target is written with `mode="w"` in the old storage format and compression, read
back and compared with the new table, and then copied byte for byte to the other target.

In [ ]:
import csv
import hashlib
import shutil


def _md5(path):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def archive_file(path, event, replaced_by, copy=False, note=""):
    """Move (or copy) path to ARCHIVE/event/<path relative to BOW_SHOCK> and add a row to INDEX.csv.
    Never overwrites. The naming rules are in ARCHIVE/README.md."""
    path = Path(path)
    rel = path.relative_to(BOW_SHOCK)
    dest = ARCHIVE / event / rel
    if dest.exists():
        raise FileExistsError(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    digest, size = _md5(path), path.stat().st_size
    modified = dt.datetime.fromtimestamp(path.stat().st_mtime).strftime("%Y-%m-%d %H:%M")
    (shutil.copy2 if copy else shutil.move)(str(path), str(dest))
    assert _md5(dest) == digest, f"archived copy differs from the original: {dest}"
    index = ARCHIVE / "INDEX.csv"
    new_index = not index.exists()
    with open(index, "a", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        if new_index:
            w.writerow(["event", "archived_path", "original_location", "file", "version_modified",
                        "archived_on", "replaced_by", "md5", "bytes", "note"])
        w.writerow([event, str(Path(event) / rel), str(rel), path.name, modified,
                    dt.datetime.now().strftime("%Y-%m-%d %H:%M"), replaced_by, digest, size, note])
    return dest


def new_event(name):
    """A folder name for this run that is not in use yet: YYYY-MM-DD_HHMM_<name>, then -2, -3 ... if needed."""
    base = f"{dt.datetime.now():%Y-%m-%d_%H%M}_{name}"
    event, k = base, 1
    while (ARCHIVE / event).exists():
        k += 1
        event = f"{base}-{k}"
    return event


if not WRITE:
    print("WRITE = False: nothing written")
else:
    event = new_event("build_hdf5_from_crossings")
    for t in TARGETS:
        if t.exists():
            print(f"archived  {archive_file(t, event, 'HDF5 rebuilt from the crossing list', copy=True)}")
    fmt = ref["fmt"] if ref else "fixed"
    kw = dict(complevel=ref["complevel"], complib=ref["complib"]) if (ref and ref["complevel"]) else {}
    first = TARGETS[0]
    first.parent.mkdir(parents=True, exist_ok=True)
    new.to_hdf(first, key=KEY, mode="w", format=fmt, **kw)
    back = pd.read_hdf(first, key=KEY)
    pd.testing.assert_frame_equal(back.reset_index(drop=True), new.reset_index(drop=True))
    print(f"wrote   {first}  ({fmt}, {len(back)} rows, md5 {md5(first)}) and read it back unchanged")
    for t in TARGETS[1:]:
        t.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(first, t)
        print(f"copied  {t}  (md5 {md5(t)})")

## Notes

- The repository's `.gitignore` ignores `*.h5` but, from release 1.2.1, carries the exception
  `!database_files/Wind_bow_shock_database.h5`, so the repository copy is committed.
- `Bow_Shock_Stats/Wind_bow_shock_database_work.h5` is a separate working file with its own columns and is not
  touched here.
- To undo, copy the archived files back over the targets; `00_old_versions/INDEX.csv` lists them.